# A detection with Urdr, step by step

This notebook first follows the signal through the EACF calculation and then runs the public joint detector. It deliberately exposes intermediate arrays for explanation; normal users only need calibrate_joint_detector(...) followed by detector.detect(series).

The current Urdr statistic filters Fourier **amplitudes** with a Tukey taper and then autocorrelates the filtered time series. By Wiener–Khinchin, the effective power-domain filter is therefore the taper squared. This is a related, window-calibrated estimator rather than an exact Lomb–Scargle reproduction of the original published EACF.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal.windows import tukey

from urdr import (
    CoherentSignalConfig,
    SegmentSystematicConfig,
    SimulationConfig,
    calibrate_joint_detector,
    compute_eacf,
    compute_eacf_map,
    joint_diagnostics,
    make_observing_window,
    simulate_time_series,
)

## 1. Define the target-specific search

In a real analysis, AsteroScale would normally provide the approximate (
u_{\max}), (Delta\nu), and envelope width. The short duration and coarse grids below keep the tutorial fast.

In [ ]:
window = make_observing_window(
    duration_days=0.8,
    cadence_seconds=120.0,
    gaps_days=((0.39, 0.41),),
)
simulation = SimulationConfig(
    white_noise_sigma=0.2,
    granulation_amplitude=0.1,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.8,
)
centres = np.linspace(700.0, 1300.0, 7)
delta_nu_grid = np.array([90.0, 100.0, 110.0])
segments = ((0.0, 0.4), (0.4, 0.8))

target = simulate_time_series(
    window,
    simulation,
    np.random.default_rng(200),
    include_oscillations=True,
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(target.time[target.observed], target.flux[target.observed], lw=0.7)
ax.axvspan(0.39, 0.41, color="0.85", label="gap")
ax.set(xlabel="Time [days]", ylabel="Flux", title="Target time series")
ax.legend()
plt.show()

## 2. Construct and apply one band-pass filter

This cell mirrors the numerical steps inside compute_eacf: robustly centre the observed flux, temporarily put zeros at missing cadences, FFT, and apply the taper. The filtered values at missing cadences are not used by the ACF.

The right panel shows both the amplitude taper (T(\nu)) and its effective power weighting (T^2(\nu)).

In [ ]:
signal = np.zeros(target.time.size)
observed_flux = target.flux[target.observed]
signal[target.observed] = observed_flux - np.median(observed_flux)

frequency_uhz = (
    np.fft.rfftfreq(signal.size, d=target.cadence_seconds) * 1e6
)
spectrum = np.fft.rfft(signal)
centre = simulation.numax_uhz
filter_width = 500.0
inside = np.abs(frequency_uhz - centre) <= filter_width / 2.0
taper_weights = np.zeros_like(frequency_uhz)
taper_weights[inside] = tukey(np.count_nonzero(inside), alpha=0.5)
filtered = np.fft.irfft(spectrum * taper_weights, n=signal.size)
filtered_for_plot = filtered.copy()
filtered_for_plot[~target.observed] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(target.time, filtered_for_plot, lw=0.8)
axes[0].axvspan(0.39, 0.41, color="0.85")
axes[0].set(
    xlabel="Time [days]",
    ylabel="Filtered flux",
    title=f"Filtered around {centre:.0f} µHz",
)
axes[1].plot(frequency_uhz, taper_weights, label=r"$T(\nu)$: amplitude")
axes[1].plot(frequency_uhz, taper_weights**2, ls="--", label=r"$T^2(\nu)$: power")
axes[1].set(
    xlim=(600, 1400),
    xlabel="Frequency [µHz]",
    ylabel="Weight",
    title="Current Urdr filter convention",
)
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Autocorrelate the filtered series

Urdr divides the raw correlation at each lag by the number of observed cadence pairs at that lag. The seismic peak is expected near (1/\Delta\nu), with supporting structure near (2/\Delta\nu).

In [ ]:
lags, acf = compute_eacf(
    target,
    centre_frequency_uhz=centre,
    filter_width_uhz=filter_width,
    max_lag_seconds=25_000.0,
)
expected_lag = 1e6 / simulation.delta_nu_uhz

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lags / 3600.0, acf)
for multiple in (1, 2):
    ax.axvline(
        multiple * expected_lag / 3600.0,
        color="tab:red",
        ls="--",
        label="expected seismic lags" if multiple == 1 else None,
    )
ax.set(xlabel="Lag [hours]", ylabel="Normalised ACF²", title="ACF at one test frequency")
ax.legend()
plt.show()

## 4. Repeat over trial frequencies

Moving the same filter through the search region gives the frequency–lag map. The joint detector computes this map once and reuses it for the EACF peak and morphology features.

In [ ]:
eacf_map = compute_eacf_map(
    target,
    centres,
    filter_width_uhz=filter_width,
    max_lag_seconds=25_000.0,
)

fig, ax = plt.subplots(figsize=(9, 5))
mesh = ax.pcolormesh(
    eacf_map.lags_seconds / 3600.0,
    eacf_map.centre_frequencies_uhz,
    eacf_map.values,
    shading="auto",
    cmap="magma",
)
for multiple in (1, 2):
    ax.axvline(multiple * expected_lag / 3600.0, color="cyan", ls="--", lw=1)
ax.axhline(simulation.numax_uhz, color="white", ls=":", lw=1)
ax.set(
    xlim=(1.5, 7.0),
    xlabel="Lag [hours]",
    ylabel="Filter centre [µHz]",
    title="Target frequency–lag EACF map",
)
fig.colorbar(mesh, ax=ax, label="Normalised ACF²")
plt.show()

## 5. Measure all diagnostic groups

joint_diagnostics selects the best trial (Delta\nu), measures spectral concentration, checks segment stability, and summarises ridge morphology. The returned 13-element feature vector is what the calibrated classifier sees.

In [ ]:
diagnostics, features = joint_diagnostics(
    series=target,
    simulation=simulation,
    centre_frequencies_uhz=centres,
    filter_width_uhz=filter_width,
    delta_nu_grid_uhz=delta_nu_grid,
    segments_days=segments,
    max_lag_seconds=25_000.0,
)
{
    "eacf_statistic": diagnostics.eacf_statistic,
    "recovered_delta_nu_uhz": diagnostics.delta_nu_uhz,
    "coherence": diagnostics.coherence,
    "segments": diagnostics.segments,
    "morphology": diagnostics.morphology,
    "feature_vector": features,
}

## 6. Calibrate against the exact observing window

Calibration simulations use the same cadence mask and compare oscillators with clean nulls, coherent signals, and segment-dependent systematics. A training subset fits one joint score; a held-out subset calibrates its false-alarm probability.

Only 16 realisations and a 25% false-positive target are used here for speed. A 1% scientific target needs at least 100 held-out negative realisations, preferably many more.

In [ ]:
detector = calibrate_joint_detector(
    window=window,
    simulation=simulation,
    centre_frequencies_uhz=centres,
    filter_width_uhz=filter_width,
    delta_nu_grid_uhz=delta_nu_grid,
    segments_days=segments,
    coherent_contaminants={
        "single_line": CoherentSignalConfig(1000.0, 0.8),
        "harmonic_comb": CoherentSignalConfig(333.3, 0.8, harmonics=3),
    },
    segment_systematics={
        "variance_jump": [
            SegmentSystematicConfig(0.4, 0.8, amplitude_scale=4.0)
        ],
    },
    realizations=16,
    validation_fraction=0.25,
    target_false_positive_rate=0.25,
    max_lag_seconds=25_000.0,
    seed=42,
)
detector.validation

## 7. Perform the detection

The final result reports a decision, a held-out-calibrated detection probability, a conformal false-alarm probability and interval, recovered (Delta\nu), and explanatory flags. The flags do not act as extra sequential vetoes.

In [ ]:
result = detector.detect(target)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(
    detector.negative_validation_scores,
    bins="auto",
    color="0.7",
    edgecolor="white",
    label="held-out hard negatives",
)
ax.axvline(result.joint_score, color="tab:blue", lw=2, label="target")
ax.axvline(detector.score_threshold, color="tab:red", ls="--", label="decision threshold")
ax.set(xlabel="Joint score", ylabel="Count", title="Target relative to calibration negatives")
ax.legend()
plt.show()

{
    "detected": result.detected,
    "probability": result.detection_probability,
    "false_alarm_probability": result.false_alarm_probability,
    "false_alarm_interval": result.false_alarm_interval,
    "joint_score": result.joint_score,
    "delta_nu_uhz": result.delta_nu_uhz,
    "flags": result.diagnostic_flags,
}